# ISOM5240 Model Comparison — Pipeline 1 & Pipeline 3
# (Pre-trained models only, NO fine-tuning)

**Pipeline 1:** Product image → Shelf-life category (short / medium / non-perishable)  
**Pipeline 3:** Product image → Auto-generated text description  

Both pipelines use pre-trained models. This notebook compares 3 candidates for each and selects the best.

## Step 1: Install dependencies

In [ ]:
!pip install transformers pillow -q

## Step 2: GPU check

In [ ]:
import torch
import os
import time
import numpy as np

if torch.cuda.is_available():
    DEVICE = 0
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = -1
    print("No GPU, using CPU (fine — no training needed)")

## Step 3: Download Grocery Store Dataset (test images)

This dataset has 81 classes of grocery products (fruits, vegetables, dairy, etc).  
We map them to shelf-life categories for testing.

In [ ]:
!git clone https://github.com/marcusklasson/GroceryStoreDataset.git
print("Downloaded!")
!ls GroceryStoreDataset/dataset/

## Step 4: Build labeled test set

In [ ]:
from PIL import Image

DATA_ROOT = "GroceryStoreDataset/dataset"

# Shelf-life mapping by folder name keywords
SHORT_KEYWORDS = [
    "apple", "avocado", "banana", "kiwi", "lemon", "lime", "mango",
    "melon", "nectarine", "orange", "papaya", "passion", "peach",
    "pear", "pineapple", "plum", "pomegranate", "grapefruit",
    "satsuma", "watermelon", "asparagus", "aubergine", "cabbage",
    "carrot", "cucumber", "garlic", "ginger", "leek", "mushroom",
    "onion", "pepper", "potato", "red-beet", "tomato", "zucchini"
]

MEDIUM_KEYWORDS = [
    "juice", "milk", "oat", "sour-cream", "sour-milk",
    "soy", "yoghurt", "cream"
]

LABEL_NAMES_P1 = ["short_shelf", "medium_shelf"]

def get_shelf_label(folder_name):
    name = folder_name.lower()
    if any(kw in name for kw in SHORT_KEYWORDS):
        return 0  # short_shelf
    elif any(kw in name for kw in MEDIUM_KEYWORDS):
        return 1  # medium_shelf
    return None  # skip unknown

# Collect test images
test_dir = os.path.join(DATA_ROOT, "test")
test_images = []
test_labels = []

for class_dir in sorted(os.listdir(test_dir)):
    class_path = os.path.join(test_dir, class_dir)
    if not os.path.isdir(class_path):
        continue

    label = get_shelf_label(class_dir)
    if label is None:
        continue

    for img_file in os.listdir(class_path):
        if img_file.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp")):
            test_images.append(os.path.join(class_path, img_file))
            test_labels.append(label)

test_labels = np.array(test_labels)
print(f"Test set: {len(test_images)} images")
print(f"  short_shelf:  {(test_labels==0).sum()}")
print(f"  medium_shelf: {(test_labels==1).sum()}")

---
# Pipeline 1: Shelf-life Classification

## Step 5: Define keyword mapping (same logic as app.py)

In [ ]:
# ImageNet label → shelf-life category
IMAGENET_SHORT = [
    "banana", "orange", "strawberry", "apple", "lemon", "pineapple",
    "pomegranate", "fig", "jackfruit", "mango", "broccoli", "cucumber",
    "mushroom", "meat", "egg", "bakery", "bread", "grocery", "fruit",
    "vegetable", "food", "pizza", "hotdog", "pretzel", "bagel", "dough",
    "zucchini", "pepper", "cauliflower", "artichoke", "potato", "cabbage",
    "head cabbage", "corn", "acorn squash", "spaghetti squash",
    "butternut squash", "ice cream", "custard apple"
]

IMAGENET_MEDIUM = [
    "bottle", "can", "jar", "packet", "carton", "sauce", "wine",
    "beer", "juice", "water", "pop", "cup", "coffee", "espresso",
    "soap", "lotion", "container", "jug", "pitcher", "milk can",
    "water bottle", "wine bottle", "beer bottle", "pop bottle"
]

def map_to_shelf_life(imagenet_label):
    label = imagenet_label.lower()
    if any(kw in label for kw in IMAGENET_SHORT):
        return 0  # short_shelf
    elif any(kw in label for kw in IMAGENET_MEDIUM):
        return 1  # medium_shelf
    else:
        return 2  # non_perishable (unmapped)

print("Keyword mapping ready")

## Step 6: Compare 3 models on Pipeline 1

In [ ]:
from transformers import pipeline as hf_pipeline
import pandas as pd

PIPELINE1_MODELS = {
    "ViT-base": "google/vit-base-patch16-224",
    "ResNet-50": "microsoft/resnet-50",
    "Swin-tiny": "microsoft/swin-tiny-patch4-window7-224",
}

def evaluate_p1(model_key, model_path, test_images, test_labels):
    print(f"\nEvaluating: {model_key}")

    pipe = hf_pipeline("image-classification", model=model_path, device=DEVICE)
    total_params = sum(p.numel() for p in pipe.model.parameters())

    correct = 0
    total = len(test_images)
    inference_times = []

    for i in range(total):
        img = Image.open(test_images[i]).convert("RGB")

        t0 = time.time()
        result = pipe(img, top_k=1)
        inference_times.append(time.time() - t0)

        pred = map_to_shelf_life(result[0]["label"])

        # Only count if the prediction is 0 or 1 (mapped successfully)
        if pred == test_labels[i]:
            correct += 1

    accuracy = correct / total
    avg_ms = np.mean(inference_times) * 1000

    print(f"  Accuracy: {accuracy:.4f} | Speed: {avg_ms:.1f}ms | Params: {total_params/1e6:.1f}M")

    return {
        "Model": model_key,
        "Parameters (M)": f"{total_params/1e6:.1f}M",
        "Accuracy": round(accuracy, 4),
        "Avg Inference (ms)": round(avg_ms, 1),
        "Test Samples": total,
    }

In [ ]:
# Run Pipeline 1 comparison
p1_results = []

for key, path in PIPELINE1_MODELS.items():
    r = evaluate_p1(key, path, test_images, test_labels)
    p1_results.append(r)

df_p1 = pd.DataFrame(p1_results)
print("\n" + "="*60)
print("PIPELINE 1 RESULTS: Shelf-life Classification")
print("="*60)
print(df_p1.to_string(index=False))

## Step 7: Select best model for Pipeline 1

In [ ]:
best_p1 = max(p1_results, key=lambda x: x["Accuracy"])
print(f"Best for Pipeline 1: {best_p1['Model']} (Accuracy: {best_p1['Accuracy']})")
print(f"\n→ Use this model in app.py: load_shelf_life_classifier()")

---
# Pipeline 3: Image Captioning

## Step 8: Compare 3 captioning models

No accuracy metric for captioning — compare inference speed + output quality.

In [ ]:
PIPELINE3_MODELS = {
    "BLIP-base": "Salesforce/blip-image-captioning-base",
    "BLIP-large": "Salesforce/blip-image-captioning-large",
    "ViT-GPT2": "nlpconnect/vit-gpt2-image-captioning",
}

def evaluate_p3(model_key, model_path, test_images, num_samples=20):
    print(f"\nEvaluating: {model_key}")

    pipe = hf_pipeline("image-to-text", model=model_path, device=DEVICE)
    total_params = sum(p.numel() for p in pipe.model.parameters())

    inference_times = []
    sample_outputs = []

    n = min(num_samples, len(test_images))
    for i in range(n):
        img = Image.open(test_images[i]).convert("RGB")
        t0 = time.time()
        result = pipe(img, max_new_tokens=50)
        inference_times.append(time.time() - t0)
        if i < 5:
            sample_outputs.append(result[0]["generated_text"])

    avg_ms = np.mean(inference_times) * 1000

    print(f"  Speed: {avg_ms:.1f}ms | Params: {total_params/1e6:.1f}M")
    print(f"  Sample outputs:")
    for j, s in enumerate(sample_outputs):
        print(f"    [{j+1}] {s}")

    return {
        "Model": model_key,
        "Parameters (M)": f"{total_params/1e6:.1f}M",
        "Avg Inference (ms)": round(avg_ms, 1),
        "Samples Tested": n,
        "Sample Output": sample_outputs[0] if sample_outputs else "",
    }

In [ ]:
# Run Pipeline 3 comparison
p3_results = []

for key, path in PIPELINE3_MODELS.items():
    r = evaluate_p3(key, path, test_images)
    p3_results.append(r)

df_p3 = pd.DataFrame(p3_results)
print("\n" + "="*60)
print("PIPELINE 3 RESULTS: Image Captioning")
print("="*60)
print(df_p3[["Model", "Parameters (M)", "Avg Inference (ms)"]].to_string(index=False))

## Step 9: Select best model for Pipeline 3

In [ ]:
# For captioning: balance speed and quality (review sample outputs above)
best_p3 = min(p3_results, key=lambda x: x["Avg Inference (ms)"])
print(f"Fastest for Pipeline 3: {best_p3['Model']} ({best_p3['Avg Inference (ms)']}ms)")
print(f"\nAlso review sample outputs above — pick the one with best quality if speeds are similar.")
print(f"→ Use selected model in app.py: load_image_captioner()")

## Step 10: Export results to Excel

In [ ]:
with pd.ExcelWriter("Pipeline1_3_Experiments.xlsx") as writer:
    df_p1.to_excel(writer, sheet_name="P1 Shelf-life", index=False)
    df_p3[["Model", "Parameters (M)", "Avg Inference (ms)", "Sample Output"]].to_excel(
        writer, sheet_name="P3 Captioning", index=False
    )

print("Saved to Pipeline1_3_Experiments.xlsx")

from google.colab import files
files.download("Pipeline1_3_Experiments.xlsx")
print("Downloaded!")

---
## Notebook Summary

| Step | Content |
|------|---------|
| 1-2 | Setup: dependencies, GPU check |
| 3-4 | Download Grocery Store Dataset, build labeled test set |
| 5-7 | **Pipeline 1:** ViT vs ResNet vs Swin + keyword mapping → accuracy + speed → select best |
| 8-9 | **Pipeline 3:** BLIP-base vs BLIP-large vs ViT-GPT2 → speed + quality → select best |
| 10 | Export Excel + auto-download |